# 加载包和处理数据
1. 我们只需要保证数据都含有content这一列（当然，数据是我处理的，肯定是有的）
2. 使用`glob`包，获得所有的数据的路径
3. 我们使用`random`包，从所有的文件路径中，随机找50个数据路径，作为训练集合的使用

In [1]:
from datasets import load_dataset, DatasetDict
from glob import glob
import random
random.seed(42)

all_file_list = glob(pathname="D:/code/geogpt/gpt2_data512/*")
test_file_list = random.sample(all_file_list, 200)
train_file_list = [i for i in all_file_list if i not in test_file_list]

len(train_file_list), len(test_file_list)

(3179, 200)

# 创建数据
1. 只要将路径放到一个字典里面。dict的key分别为`train`、`valid`，他们对应的值就是文件路径列表即可

In [2]:
raw_datasets =load_dataset("csv",data_files={'train':train_file_list,'valid':test_file_list}, cache_dir="cache_data")

raw_datasets

Resolving data files:   0%|          | 0/3179 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/200 [00:00<?, ?it/s]

Found cached dataset csv (D:/code/geogpt/cache_data/csv/default-9d7e37516925fd5d/0.0.0/6954658bab30a358235fa864b05cf819af0e179325c740e4bc853bcc7ec513e1)


  0%|          | 0/2 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['content'],
        num_rows: 99890
    })
    valid: Dataset({
        features: ['content'],
        num_rows: 3403
    })
})

# Tokenizer
1. Tokenizer 是最关键的一步，因为我们处理的是中文，因此使用`bert_base_chinese`就足够了
2. 如果你的语料里面有别的语言，你也可以使用多语言。这个都无所谓的。只要保证你使用的Tokenizer能覆盖你的数据即可
3. `context_length = 512`设置你的每一个文本的最长长度，我这里设置的是512，如果你的显卡显寸小，那你可以改小一点，比如128。但是多出来的数据，并不是说直接截断不要了，而是按照`context_length`长度，不断的对文本进行截断，大概就像是下面这样的：

<img src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter7/chunking_texts.svg"/>


4. 对于`gpt2`模型，需要告诉模型一句话从哪里开始，从哪里结束。因此我们需要设置`bos_token`、`eos_token`、`unk_token`



In [3]:
from transformers import AutoTokenizer, AutoConfig

context_length = 128
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

outputs = tokenizer(
    raw_datasets["train"][:2]["content"],
    truncation=True,
    max_length=context_length,
    return_overflowing_tokens=True,
    return_length=True,
)

print(f"Input IDs length: {len(outputs['input_ids'])}")
print(f"Input chunk lengths: {(outputs['length'])}")
print(f"Chunk mapping: {outputs['overflow_to_sample_mapping']}")

'HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-chinese/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000172303C3EE0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))' thrown while requesting HEAD https://huggingface.co/bert-base-chinese/resolve/main/tokenizer_config.json


Input IDs length: 10
Input chunk lengths: [128, 128, 128, 128, 10, 128, 128, 128, 128, 10]
Chunk mapping: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]


In [4]:
tokenizer.add_special_tokens(special_tokens_dict={'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'})

1

# 说明
1. 其实，到这里，我做的东西基本上就结束了。
2. 只要查看这个链接，说明的更加清楚：[https://huggingface.co/course/zh-CN/chapter7/6?fw=pt](https://huggingface.co/course/zh-CN/chapter7/6?fw=pt),我也就是模仿这个链接的。

In [5]:
def tokenize(element):
    outputs = tokenizer(
        element["content"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}


tokenized_datasets = raw_datasets.map(
    tokenize, batched=True, remove_columns=raw_datasets["train"].column_names
)
tokenized_datasets

Loading cached processed dataset at D:\code\geogpt\cache_data\csv\default-9d7e37516925fd5d\0.0.0\6954658bab30a358235fa864b05cf819af0e179325c740e4bc853bcc7ec513e1\cache-b9744bfe4526c8cb.arrow
Loading cached processed dataset at D:\code\geogpt\cache_data\csv\default-9d7e37516925fd5d\0.0.0\6954658bab30a358235fa864b05cf819af0e179325c740e4bc853bcc7ec513e1\cache-f0bbe37a5059c7c8.arrow


DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 351896
    })
    valid: Dataset({
        features: ['input_ids'],
        num_rows: 13113
    })
})

# 模型
1. 这里的`gpt2`模型，可不是使用别人训练好的，就是一个gpt2配置，因为我们要使用这个从头训练一个新的`gpt2`

In [6]:
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig


##
GPT_MODEL_NAME_OR_PATH = "yuanzhoulvpi/gpt2_chinese"
GPT_model = GPT2LMHeadModel.from_pretrained(GPT_MODEL_NAME_OR_PATH, add_cross_attention=True)
##

config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=context_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

'HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /yuanzhoulvpi/gpt2_chinese/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000017256AF3730>, 'Connection to huggingface.co timed out. (connect timeout=10)'))' thrown while requesting HEAD https://huggingface.co/yuanzhoulvpi/gpt2_chinese/resolve/main/config.json
Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at yuanzhoulvpi/gpt2_chinese and are newly initialized: ['transformer.h.1.ln_cross_attn.weight', 'transformer.h.5.crossattention.bias', 'transformer.h.11.crossattention.c_attn.weight', 'transformer.h.9.crossattention.c_proj.weight', 'transformer.h.0.crossattention.masked_bias', 'transformer.h.5.crossattention.c_attn.weight', 'transformer.h.9.crossattention.bias', 'transformer.h.5.ln_cross_attn.weight', 'transformer.h.6.crossattention.q_attn.weight', 'transformer.h.4.crossattention.c_proj.weight', 'transformer.h

In [7]:
model = GPT_model
#model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 size: {model_size/1000**2:.1f}M parameters")

GPT-2 size: 130.4M parameters


In [8]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

KeyboardInterrupt: 

In [ ]:
out = data_collator([tokenized_datasets["train"][i] for i in range(5)])
for key in out:
    print(f"{key} shape: {out[key].shape}")

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="gpt2_geo_test",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    evaluation_strategy="steps",
    eval_steps=4000,
    logging_steps=4000,
    logging_dir='logs_gpt2_geo_test',
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.1,
    warmup_steps=2000,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    save_steps=4000,
    fp16=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
)

In [ ]:
trainer.train()